In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Summarize messages

In [2]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [3]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user is inquiring about various aspects of Lunapolis, a fictional location on the moon, including its capital, weather, population of cheese miners, and potential labor actions.\n\n## SUMMARY\n- The capital of the moon is identified as Lunapolis.\n- The weather in Lunapolis is described as clear skies with extreme temperatures: a high of 120°C and a low of -100°C.\n- The population of cheese miners in Lunapolis is stated to be 100,000.\n- It is anticipated that the cheese miners' union may go on strike due to dissatisfaction with the new president.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nNone", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='116f502c-4edc-4766-93dc-41dd01c39e59'),
              HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?", additional_kwargs={}, response_metadata={}, id=

In [4]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT
The user is inquiring about various aspects of Lunapolis, a fictional location on the moon, including its capital, weather, population of cheese miners, and potential labor actions.

## SUMMARY
- The capital of the moon is identified as Lunapolis.
- The weather in Lunapolis is described as clear skies with extreme temperatures: a high of 120°C and a low of -100°C.
- The population of cheese miners in Lunapolis is stated to be 100,000.
- It is anticipated that the cheese miners' union may go on strike due to dissatisfaction with the new president.

## ARTIFACTS
None

## NEXT STEPS
None


## Trim/delete messages

Middleware provides two styles of hooks to intercept agent execution : Node-style and wrap-style

- `before_agent` - Before agent starts (once per invocation) - Node-Style hooks
- `before_model` - Before each model call
- `after_model` - After each model response
- `after_agent` - After agent completes (once per invocation)


In [5]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [6]:
agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [7]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='20c3fdaa-8a77-4d09-bba6-ad17b5b3e71c'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='9fa8d1ed-aa4e-46a4-b15d-16176d079057', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='71347371-2548-4e4b-86b4-573e2b1299f1'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='166a64f0-8484-4715-b6bd-ce2124e0ba93', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='130936ef-fb16-421a-b661-55d5b9420fdf'),
              AIMessage(content='I can’t read your device’s temperature from here. If you want 

In [8]:
print(response["messages"][-1].content)

I can’t read your device’s temperature from here. If you want to check the temperature yourself, you’d need to monitor it with the device powered or use a tool after it boots. Since it won’t turn on, here are safe steps you can take related to heat:

- Safety first: If the device feels very hot or you smell burning, unplug it and let it cool in a well-ventilated area for 15–30 minutes before trying again.
- Check for obvious signs of overheating: blocked vents, dust buildup, swollen battery (for laptops/phones), or a burnt smell.
- Try a safe power reset:
  - Laptop: unplug, remove the battery if possible, hold the power button for 15–20 seconds, reconnect power (and battery if removable) and try powering on.
  - Desktop: unplug, hold power for 15 seconds, reconnect and try again.
- Test with a known-good power source or charger (if removable battery is present, try without the battery using AC power only, if manufacturer allows).
- Boot with minimal peripherals: remove USB devices, ex